# [SK 03 - From `get_chat_message_contents` to `ChatCompletion*Agent*`](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)
## WITH kernel, but NO tools
Now we learn how to create a first-class agent, with no tool (=plugin) for the moment.

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")

agent_name                = "agent_name"
chatcompletion_service_id = "chatcompletion_service_id"
instructions              = "you are a clever agent"
content                   = "tell me what is Azure in less than 10 words"

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [2]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler())
logging.getLogger().setLevel(logging.ERROR) # or logging.DEBUG to see more details

# Create the`AzureChatCompletion` object, and add it to the Kernel just created

In [3]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x77d30e5f2120>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x77d30e904ad0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Get ready to print all messages of a [`ChatHistoryAgentThread`](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/examples/example-chat-agent?pivots=programming-language-python) object
The history is managed through a `thread` object.
<br/><br/>
The `async for` syntax is specifically designed to iterate over asynchronous generators. It handles the yielding of items properly and allows you to collect them into a list or process them one by one.<br/>
By contrast, `await` is used for simple coroutines that return a single result, not for iterating through async generators.es.

In [4]:
from semantic_kernel.agents import ChatHistoryAgentThread
import asyncio

async def print_messages(thread: ChatHistoryAgentThread):
    i=0
    messages = thread.get_messages()
    async for cmc in messages: # ChatMessageContent
        i += 1
        print(f"{i} - role: {cmc.role.value}, text: {cmc.items[0].text}")

# Create the `agent` instead of using `get_chat_message_content`
The agent `service_id` specified in `ChatCompletionAgent` must match one of the services defined in `Kernel.services`

In [5]:
from semantic_kernel.agents import ChatCompletionAgent

agent = ChatCompletionAgent(
    name=agent_name,
    instructions=instructions,
    kernel=kernel
)

agent

ChatCompletionAgent(arguments=None, description=None, id='d0113059-a8b9-4f48-81dd-a54a15e22578', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x77d30e5f2120>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x77d30e904ad0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='agent_name', prompt_template=None, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), service=None)

# Generate the agent response using `invoke`
`agent.invoke` performs an asynchronous operation.<br/>
Because the responses arrive gradually, we use `async for` to efficiently process them without blocking execution.

In [6]:
from semantic_kernel.functions import KernelArguments

thread: ChatHistoryAgentThread = None

async for response in agent.invoke(messages=content, thread=thread):
    thread = response.thread
    await print_messages(thread)

print(f"\nRESPONSE:\n{response}")

1 - role: user, text: tell me what is Azure in less than 10 words
2 - role: assistant, text: Microsoft's cloud platform for services and computing resources.

RESPONSE:
Microsoft's cloud platform for services and computing resources.


# Generate a follow-up question for the thread

In [7]:
async for response in agent.invoke(messages="translate your answer to Italian", thread=thread):
    thread = response.thread
    messages = thread.get_messages()
    await print_messages(thread)

print(f"\nRESPONSE:\n{response}")

1 - role: user, text: tell me what is Azure in less than 10 words
2 - role: assistant, text: Microsoft's cloud platform for services and computing resources.
3 - role: user, text: translate your answer to Italian
4 - role: assistant, text: La piattaforma cloud di Microsoft per servizi e risorse informatiche.

RESPONSE:
La piattaforma cloud di Microsoft per servizi e risorse informatiche.


# TEARDOWN
No resources to de-allocate here, since ChatCompletion agents are just in-memory objects bound to the kernel.

In [8]:
agent.model_dump()

{'arguments': None,
 'description': None,
 'id': 'd0113059-a8b9-4f48-81dd-a54a15e22578',
 'instructions': 'you are a clever agent',
 'kernel': {'services': {'chatcompletion_service_id': {'ai_model_id': 'gpt-4o',
    'service_id': 'chatcompletion_service_id'}},
  'ai_service_selector': <semantic_kernel.services.ai_service_selector.AIServiceSelector at 0x77d30e904ad0>,
  'plugins': {},
  'function_invocation_filters': [],
  'prompt_rendering_filters': [],
  'auto_function_invocation_filters': []},
 'name': 'agent_name',
 'prompt_template': {'prompt_template_config': {'name': '',
   'description': '',
   'template': 'you are a clever agent',
   'template_format': 'semantic-kernel',
   'input_variables': [],
   'allow_dangerously_set_content': False,
   'execution_settings': {}},
  'allow_dangerously_set_content': False},
 'function_choice_behavior': {'enable_kernel_functions': True,
  'maximum_auto_invoke_attempts': 5,
  'filters': None,
  'type_': <FunctionChoiceType.AUTO: 'auto'>}}